# Transform Results Data
1. Read bronze `results` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`constructorId` → `constructor_id`, `driverId` → `driver_id`, `raceName` → `race_name`, `positionText` → `finish_position_text`)
1. Rename columns to make them more meaningful (`date` → `race_date`, `grid` → `grid_position`, `laps` → `completed_laps`, `number` → `car_number`, `position` → `finish_position`)
1. Filter out rows where `season`, `round`, `custructor_id` or `driver_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of column `race_name` to Title Case
1. Write the transformed data to silver `results` table

#### Entity Relationship Diagram - Formula1 Bronze Schema
![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")



In [0]:
%run "../00-common/01.environment-config"

In [0]:
val bronze_table = catalog_name + "." + bronze_schema + "." + "results "
val silver_table = catalog_name + "." + silver_schema + "." + "results"

#### Step 1 to 7 - Read , transform, & perform data quality checks

In [0]:
import org.apache.spark.sql.functions.{col,initcap}
val results_df=spark.table(bronze_table)
.select("season","round", "constructorId","driverId", "date","raceName","grid","laps", "number","points","position","positionText","status","ingestion_timestamp","source_file")
.withColumnRenamed("constructorId", "constructor_id")
.withColumnRenamed("driverId", "driver_id")
.withColumnRenamed("raceName", "race_name")
.withColumnRenamed("date", "race_date")
.withColumnRenamed("grid", "grid_position")
.withColumnRenamed("laps", "completed_laps")
.withColumnRenamed("number", "car_number")
.withColumnRenamed("position", "final_position")
.withColumnRenamed("positionText", "final_position_text")
.filter(
  col("season").isNotNull &&
  col("round").isNotNull &&
  col("constructor_id").isNotNull &&
  col("driver_id").isNotNull)
.dropDuplicates("season", "round", "constructor_id", "driver_id")
.withColumn("race_name", initcap(col("race_name")))


#### Step 8 - Write the transformed data to silver `results` table

In [0]:
results_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)